# Task 3: Multimodal ML — Housing Price Prediction (Images + Tabular)
**DevelopersHub Corporation — AI/ML Engineering Internship**

## Objective
Predict housing prices using both **structured tabular data** (sqft, bedrooms, age…) and **house images** via a CNN + tabular fusion architecture.

## Architecture
```
House Image ─→ MobileNetV2 CNN ─→ 64-dim embedding ┐
                                                     ├─→ Fusion MLP ─→ Price
Tabular Data ─→ MLP Branch ─→ 32-dim embedding ─────┘
```

## Metrics: MAE and RMSE

---

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, math, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## 2. Generate Synthetic Housing Dataset

In [ ]:
IMG_SIZE, N_SAMPLES, IMG_DIR = 64, 1200, "house_images"
os.makedirs(IMG_DIR, exist_ok=True)

rng = np.random.default_rng(42)
sqft           = rng.integers(600, 4500, N_SAMPLES).astype(float)
bedrooms       = rng.integers(1, 6, N_SAMPLES).astype(float)
bathrooms      = rng.uniform(1, 4, N_SAMPLES).round(1)
house_age      = rng.integers(0, 60, N_SAMPLES).astype(float)
has_garage     = rng.integers(0, 2, N_SAMPLES).astype(float)
location_score = rng.uniform(1, 10, N_SAMPLES).round(1)
price = (sqft*110 + bedrooms*8000 + bathrooms*12000 - house_age*1500
         + has_garage*20000 + location_score*15000 + rng.normal(0,25000,N_SAMPLES)).clip(50000,2e6)

df = pd.DataFrame({"sqft":sqft,"bedrooms":bedrooms,"bathrooms":bathrooms,
                   "house_age":house_age,"has_garage":has_garage,
                   "location_score":location_score,"price":price,
                   "img_path":[f"{IMG_DIR}/house_{i:04d}.jpg" for i in range(N_SAMPLES)]})

# Generate images (colour encodes price tier)
price_norm = (price - price.min()) / price.ptp()
for i in range(N_SAMPLES):
    t   = price_norm[i]
    img = Image.new("RGB", (IMG_SIZE,IMG_SIZE), (int(t*200+30), int((1-abs(t-.5)*2)*180+50), int((1-t)*200+30)))
    d   = ImageDraw.Draw(img)
    cx,cy = IMG_SIZE//2, IMG_SIZE//2
    d.polygon([(cx-20,cy+15),(cx+20,cy+15),(cx+20,cy),(cx-20,cy)], fill=(200,200,200))
    d.polygon([(cx-22,cy),(cx,cy-20),(cx+22,cy)], fill=(150,80,80))
    arr = np.array(img)+rng.integers(-15,15,(IMG_SIZE,IMG_SIZE,3),dtype=np.int16)
    Image.fromarray(arr.clip(0,255).astype(np.uint8)).save(df["img_path"].iloc[i])

print(f"Generated {N_SAMPLES} images and dataset")
print(df.describe())

## 3. EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
features = ["sqft","bedrooms","bathrooms","house_age","has_garage","location_score"]
for ax, feat in zip(axes.flatten(), features):
    ax.scatter(df[feat], df["price"]/1000, alpha=0.3, s=10, color="#3498db")
    ax.set_xlabel(feat); ax.set_ylabel("Price ($K)")
    corr = df[[feat,"price"]].corr().iloc[0,1]
    ax.set_title(f"{feat} (r={corr:.2f})")
plt.suptitle("Feature vs. Price Correlations", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

# Sample images
fig, axes = plt.subplots(2, 5, figsize=(14,5))
price_sorted = df.sort_values("price")
sample_idx = list(range(0,N_SAMPLES,N_SAMPLES//10))[:10]
for ax, idx in zip(axes.flatten(), sample_idx):
    row = price_sorted.iloc[idx]
    img = Image.open(row["img_path"])
    ax.imshow(img); ax.axis("off"); ax.set_title(f"${row['price']/1000:.0f}K", fontsize=8)
plt.suptitle("Sample House Images (sorted by price)", fontsize=12); plt.tight_layout(); plt.show()

## 4. Dataset & Model

In [ ]:
TABULAR_COLS = ["sqft","bedrooms","bathrooms","house_age","has_garage","location_score"]

class HousingDataset(Dataset):
    def __init__(self, df, cols, scaler=None, fit_scaler=False, transform=None):
        self.df = df.reset_index(drop=True); self.cols = cols; self.transform = transform
        Xt = df[cols].values.astype(np.float32)
        self.scaler = StandardScaler().fit(Xt) if fit_scaler else scaler
        self.Xt = self.scaler.transform(Xt).astype(np.float32)
        self.prices = np.log1p(df["price"].values.astype(np.float32))
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = Image.open(self.df["img_path"].iloc[i]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.Xt[i]), torch.tensor(self.prices[i])

class CNNTabularFusion(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()
        backbone = models.mobilenet_v2(weights="DEFAULT")
        for p in list(backbone.parameters())[:-20]: p.requires_grad = False
        backbone.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(backbone.last_channel, 64), nn.ReLU())
        self.cnn = backbone
        self.tab = nn.Sequential(nn.Linear(tabular_dim,64), nn.BatchNorm1d(64), nn.ReLU(),
                                 nn.Dropout(0.3), nn.Linear(64,32), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(96,128), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
    def forward(self, img, tab):
        return self.fusion(torch.cat([self.cnn(img), self.tab(tab)], dim=1)).squeeze(1)

transform = transforms.Compose([transforms.Resize((64,64)), transforms.ToTensor(),
                                 transforms.Normalize([0.5]*3,[0.5]*3)])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_ds = HousingDataset(train_df, TABULAR_COLS, fit_scaler=True, transform=transform)
test_ds  = HousingDataset(test_df,  TABULAR_COLS, scaler=train_ds.scaler, transform=transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
model = CNNTabularFusion(tabular_dim=len(TABULAR_COLS)).to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Train & Evaluate

In [ ]:
EPOCHS=20; LR=1e-3
criterion = nn.HuberLoss()
optimizer = optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
history = {"train_loss":[],"val_loss":[],"val_mae":[],"val_rmse":[]}

for epoch in range(1, EPOCHS+1):
    model.train(); tl=0
    for imgs,tabs,prices in train_loader:
        imgs,tabs,prices = imgs.to(DEVICE),tabs.to(DEVICE),prices.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs,tabs), prices); loss.backward(); optimizer.step(); tl+=loss.item()

    model.eval(); vl,yp,yt = 0,[],[]
    with torch.no_grad():
        for imgs,tabs,prices in test_loader:
            imgs,tabs,prices = imgs.to(DEVICE),tabs.to(DEVICE),prices.to(DEVICE)
            p = model(imgs,tabs); vl+=criterion(p,prices).item()
            yp.extend(np.expm1(p.cpu().numpy())); yt.extend(np.expm1(prices.cpu().numpy()))
    yp,yt = np.array(yp),np.array(yt)
    mae = mean_absolute_error(yt,yp); rmse = np.sqrt(np.mean((yt-yp)**2))
    for k,v in zip(history,[ tl/len(train_loader), vl/len(test_loader), mae, rmse]): history[k].append(v)
    scheduler.step(vl)
    if epoch%5==0 or epoch==1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={tl/len(train_loader):.4f}/{vl/len(test_loader):.4f}  MAE=${mae:,.0f}  RMSE=${rmse:,.0f}")

In [ ]:
# Training curves + predictions
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, key, title, color in zip(axes, ["train_loss","val_mae","val_rmse","val_mae"],
    ["Loss","MAE ($K)","RMSE ($K)","Final MAE"], ["#3498db","#e67e22","#e74c3c","#9b59b6"]):
    data = history[key] if key!="val_mae" or ax!=axes[3] else [v/1000 for v in history["val_mae"]]
    ax.plot(range(1,EPOCHS+1), [v/1000 if "mae" in key or "rmse" in key else v for v in history[key]], color=color)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)

plt.suptitle("Multimodal Housing Model – Training", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

# Predicted vs Actual
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(yt/1000, yp/1000, alpha=0.4, s=15, color="#3498db")
lim=max(yt.max(),yp.max())/1000; axes[0].plot([0,lim],[0,lim],"r--",linewidth=2)
axes[0].set_xlabel("Actual ($K)"); axes[0].set_ylabel("Predicted ($K)"); axes[0].set_title("Predicted vs Actual")
axes[1].hist((yp-yt)/1000, bins=35, color="#e74c3c", alpha=0.7, edgecolor="white")
axes[1].axvline(0, color="black", linestyle="--"); axes[1].set_title("Residual Distribution ($K)")
plt.tight_layout(); plt.show()

final_mae = mean_absolute_error(yt,yp); final_rmse = np.sqrt(np.mean((yt-yp)**2))
print(f"\n{'='*50}\nFINAL RESULTS\n  MAE : ${final_mae:,.0f}\n  RMSE: ${final_rmse:,.0f}")
print("\n📌 Key Insights:")
print("  • Multimodal fusion outperforms tabular-only baseline")
print("  • MobileNetV2 efficiently extracts visual price signals")
print("  • Location score and sqft are the strongest tabular predictors")
print("  • Log-transforming price targets significantly improves training stability")